# Data cleaning exercise
Data is from ChatGPT, it is messy. .csv file has inconsistent quotes (', "), mixed casing (USA, us, germany, etc.), missing values ("", None), inconsistent date formats, mixed state formats (California, CA, florida, etc.), numeric columns with noise and missing entries. 
<br></br>
Task is to perform data cleaning handling, formating, missing values, etc...

In [ ]:
import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt

set_data = pd.read_csv('/Users/gabrielaslomiany/PyDeveloper/DataCleaning/set.csv')
set_df = pd.DataFrame(set_data)
print(set_df.head(15))

: 

In [ ]:
print(set_df.info())

In [ ]:
print(set_df.describe())

In [ ]:
print(set_df.columns)

In [ ]:
print(set_df.dtypes)

In [ ]:
print(set_df.isna().sum())

## Observations of the dirty df
<ol>
    <li>Problem with names, capitalized, small case, in different quotations, couple of records has every value in one cell</li>
    <li>Missing values represented as NaN --> change it to --- for clearer look</li>
    <li>Gender letters should be capitalized, missing values represented as --- , later new binary field should be added to represent gender it could be helpful for some ML</li>
    <li>Emails are sometimes capitalized, sometimes missing .com - if such email is found should be either removed or replaced with Unknown, since we cannot predict ending of the email adress</li>
    <li>phone numbers are represented either with - or without, additional field for country number should be added if number has it</li>
    <li>Names of the countries sould be the corrected</li>
    <li>Age should be represented as type int, not str</li>
    <li>Loyalty points change to int, missing values set 0 by default</li>
    <li>Date should be standardised by ISO format</li>
</ol>

## Cleansing

In [ ]:
clean_data = pd.read_csv('/Users/gabrielaslomiany/PyDeveloper/DataCleaning/set.csv')
clean_df = pd.DataFrame(clean_data)

print(clean_df.columns)

In [ ]:
clean_df.dropna(inplace=True) #inplace=True --> Whether to modify the DataFrame rather than creating a new one
print(clean_df.isna().sum())

In [ ]:
print(clean_df.shape)

## Standarization of text

In [ ]:
# insufficient, works but is not the best idea when working on +10 fields
# clean_df['first_name'] = clean_df['first_name'].str.capitalize()
# clean_df['last_name'] = clean_df['last_name'].str.capitalize()

name = ['first_name', 'last_name']
for n in name:
    clean_df[n] = clean_df[n].str.capitalize()
    
print(clean_df[['first_name', 'last_name']].head(10))

In [ ]:
clean_df['gender'] = clean_df['gender'].str.upper()
# clean_df['gender'] = clean_df['gender'].fillna('---')
# since I've decided to drop na earlier I do not have to deal with it now 

clean_df['binary_gender'] = clean_df['gender'].apply(lambda x: 1 if x=='F' else 0)
clean_df['binary_gender'] = clean_df['binary_gender'].astype(int)
print(clean_df[['gender', 'binary_gender']].head(10))

In [ ]:
clean_df['email'] = clean_df['email'].str.lower()
print(clean_df['email'].head(15))

# we can see the record 10 and 22 does not have the .com ; since there is a lot of email endings we cannot predict what it will be or simply add .com, so in my opionion best practice 
# would be to check wheter email contains certain characters in order and if not replace it with Unknown

In [ ]:
clean_df['email_valid'] = clean_df['email'].str.contains(r'@.+\.', na=False)
clean_df.loc[~clean_df['email_valid'], 'email'] = 'Unknown'
print(clean_df['email'].head(15))

# right now we can see that records 10 and 22 were changed to Unknown

In [ ]:
# just by looking at the phone field, we can say that ther are two ways to write the phone number with and without - ; let's change it and remove it using .strip()
clean_df['phone'] = clean_df['phone'].str.replace('-', '')
clean_df['phone'].head(15)

In [ ]:
clean_df['country'] = clean_df['country'].str.capitalize()
USA = ['Usa', 'Us', 'United states'] 
clean_df['country'] = clean_df['country'].replace(USA, 'United States of America')

print(clean_df['country'].head(15))

In [ ]:
usa_states = {
    'AL': 'Alabama', 'AK': 'Alaska', 'AZ': 'Arizona', 'AR': 'Arkansas',
    'CA': 'California', 'CO': 'Colorado', 'CT': 'Connecticut', 'DE': 'Delaware',
    'FL': 'Florida', 'GA': 'Georgia', 'HI': 'Hawaii', 'ID': 'Idaho',
    'IL': 'Illinois', 'IN': 'Indiana', 'IA': 'Iowa', 'KS': 'Kansas',
    'KY': 'Kentucky', 'LA': 'Louisiana', 'ME': 'Maine', 'MD': 'Maryland',
    'MA': 'Massachusetts', 'MI': 'Michigan', 'MN': 'Minnesota', 'MS': 'Mississippi',
    'MO': 'Missouri', 'MT': 'Montana', 'NE': 'Nebraska', 'NV': 'Nevada',
    'NH': 'New Hampshire', 'NJ': 'New Jersey', 'NM': 'New Mexico', 'NY': 'New York',
    'NC': 'North Carolina', 'ND': 'North Dakota', 'OH': 'Ohio', 'OK': 'Oklahoma',
    'OR': 'Oregon', 'PA': 'Pennsylvania', 'RI': 'Rhode Island', 'SC': 'South Carolina',
    'SD': 'South Dakota', 'TN': 'Tennessee', 'TX': 'Texas', 'UT': 'Utah',
    'VT': 'Vermont', 'VA': 'Virginia', 'WA': 'Washington', 'WV': 'West Virginia',
    'WI': 'Wisconsin', 'WY': 'Wyoming'
}

canada_provinces = {
    'AB': 'Alberta', 'BC': 'British Columbia', 'MB': 'Manitoba',
    'NB': 'New Brunswick', 'NL': 'Newfoundland and Labrador',
    'NS': 'Nova Scotia', 'NT': 'Northwest Territories', 'NU': 'Nunavut',
    'ON': 'Ontario', 'PE': 'Prince Edward Island', 'QC': 'Quebec',
    'SK': 'Saskatchewan', 'YT': 'Yukon'
}

mexico_states = {
    'CDMX': 'Ciudad de Mexico',
    'Jalisco': 'Jalisco',
    'Puebla': 'Puebla',
    'Sonora': 'Sonora',
    'Chiapas': 'Chiapas',
    'Durango': 'Durango',
    'Guerrero': 'Guerrero',
    'Oaxaca': 'Oaxaca',
    'Yucatan': 'Yucatan',
    'Nuevo Leon': 'Nuevo Leon'
}


clean_df['state'] = clean_df['state'].replace(usa_states)
clean_df['state'] = clean_df['state'].replace(canada_provinces)
clean_df['state'] = clean_df['state'].replace(mexico_states)
clean_df['state'].head(20)

In [ ]:
# clean_df['age'].head(39)

numbers_to_words = {
"ten": 10, "eleven": 11, "twelve": 12, "thirteen": 13, "fourteen": 14,
"fifteen": 15, "sixteen": 16, "seventeen": 17, "eighteen": 18, "nineteen": 19,
"twenty": 20, "twenty one": 21, "twenty two": 22, "twenty three": 23, "twenty four": 24,
"twenty five": 25, "twenty six": 26, "twenty seven": 27, "twenty eight": 28, "twenty nine": 29,
"thirty": 30, "thirty one": 31, "thirty two": 32, "thirty three": 33, "thirty four": 34,
"thirty five": 35, "thirty six": 36, "thirty seven": 37, "thirty eight": 38, "thirty nine": 39,
"forty": 40, "forty one": 41, "forty two": 42, "forty three": 43, "forty four": 44,
"forty five": 45, "forty six": 46, "forty seven": 47, "forty eight": 48, "forty nine": 49,
"fifty": 50, "fifty one": 51, "fifty two": 52, "fifty three": 53, "fifty four": 54,
"fifty five": 55, "fifty six": 56, "fifty seven": 57, "fifty eight": 58, "fifty nine": 59,
"sixty": 60, "sixty one": 61, "sixty two": 62, "sixty three": 63, "sixty four": 64,
"sixty five": 65, "sixty six": 66, "sixty seven": 67, "sixty eight": 68, "sixty nine": 69,
"seventy": 70, "seventy one": 71, "seventy two": 72, "seventy three": 73, "seventy four": 74,
"seventy five": 75, "seventy six": 76, "seventy seven": 77, "seventy eight": 78, "seventy nine": 79,
"eighty": 80, "eighty one": 81, "eighty two": 82, "eighty three": 83, "eighty four": 84,
"eighty five": 85, "eighty six": 86, "eighty seven": 87, "eighty eight": 88, "eighty nine": 89,
"ninety": 90, "ninety one": 91, "ninety two": 92, "ninety three": 93, "ninety four": 94,
"ninety five": 95, "ninety six": 96, "ninety seven": 97, "ninety eight": 98, "ninety nine": 99,
"one hundred": 100
}

clean_df['age'] = clean_df['age'].astype(str).str.strip().str.lower()
clean_df['age'] = clean_df['age'].replace(numbers_to_words)

# convert to numeric (invalid → NaN)
clean_df['age'] = pd.to_numeric(clean_df['age'], errors='coerce')

# fill na with 0
clean_df['age'] = clean_df['age'].fillna(0)
clean_df['age'] = clean_df['age'].astype(int)
clean_df['age'].dtypes

In [ ]:
clean_df['loyalty_points'] = clean_df['loyalty_points'].astype(int)
clean_df['loyalty_points'].dtypes
clean_df['loyalty_points'].head()

In [ ]:
# I would like to change all dates to ISO 8601 format YYYY-MM-DD
import datetime
clean_df['date_joining'] = pd.to_datetime(clean_df['date_joining'], errors='coerce')
# errors='coerce' in Pandas is a data cleaning parameter that tells functions (like to_numeric or to_datetime) to convert invalid parsing data into missing values (NaN or NaT) 
# rather than raising an error or skipping the entry. This allows the script to continue running despite faulty data. 

clean_df['date_joining'].head(15)

In [ ]:
clean_df.head(39)

In [ ]:
clean_df.to_csv('clean_data.csv')